In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock
import pandas as pd
import re
import time

open_ai_keys = [
    os.getenv("OPENAI_KEY_1"),
    os.getenv("OPENAI_KEY_2"),
    os.getenv("OPENAI_KEY_3"),
    os.getenv("OPENAI_KEY_4"),
    os.getenv("OPENAI_KEY_5"),
    os.getenv("OPENAI_KEY_6"),
    os.getenv("OPENAI_KEY_7"),
    os.getenv("OPENAI_KEY_8"),
    os.getenv("OPENAI_KEY_9"),
    os.getenv("OPENAI_KEY_10"),
    os.getenv("OPENAI_KEY_11"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_13"),
    os.getenv("OPENAI_KEY_14"),
    os.getenv("OPENAI_KEY_15"),
    os.getenv("OPENAI_KEY_16"),
    os.getenv("OPENAI_KEY_17"),
    os.getenv("OPENAI_KEY_18"),
    os.getenv("OPENAI_KEY_19"),
    os.getenv("OPENAI_KEY_20"),
    os.getenv("OPENAI_KEY_21"),
    os.getenv("OPENAI_KEY_22"),
    os.getenv("OPENAI_KEY_23"),
    os.getenv("OPENAI_KEY_24"),
    os.getenv("OPENAI_KEY_25")
]

In [2]:

executor = ThreadPoolExecutor(max_workers=25)

In [3]:
transcript = "../runs/2008-09-26-debate-preclean.txt"

In [6]:
def process_transcript(file_path):
    # Open the file.
    with open(file_path, "r") as file:
        # Read the content.
        content = file.read()

    # Split the content into lines.
    lines = content.split("\n")

    # Initialize a variable to keep track of the current speaker.
    current_speaker = None

    # Initialize a variable to keep track of the speaker's turn.
    turn = 0

    # Prepare a list to store the rows of the DataFrame.
    rows = []

    # Initialize a variable to keep track of the current line.
    current_line = ""

    # Go through each line.
    for line in lines:
        # Check if line contains the speaker's name and what they say.
        if ": " in line:
            # Split the line into the speaker's name and what they say.
            speaker, line = line.split(": ", 1)

            # If the speaker has changed, add the previous turn to the list of rows and start a new turn.
            if speaker != current_speaker:
                if current_speaker is not None:
                    rows.append([turn, current_speaker, current_line.strip()])
                turn += 1
                current_speaker = speaker
                current_line = line
            else:
                # If the speaker hasn't changed, add the line to the current turn.
                current_line += " " + line
        else:
            # If the line doesn't contain a speaker, add it to the current turn.
            current_line += " " + line

    # Add the last turn to the list of rows.
    rows.append([turn, current_speaker, current_line.strip()])

    # Create a DataFrame from the list of rows.
    df = pd.DataFrame(rows, columns=["Turn", "Speaker", "Content"])

    # Return the DataFrame.
    return df


In [9]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', None)

In [7]:
transcript_dataframe = process_transcript(transcript)
transcript_dataframe

,Turn,Speaker,Content
0,1,LEHRER,Good evening from the Ford Center for the Perf...
1,2,OBAMA,"Well, thank you very much, Jim, and thanks to ..."
2,3,LEHRER,"Senator McCain, two minutes."
3,4,MCCAIN,"Well, thank you, Jim. And thanks to everybody...."
4,5,LEHRER,"All right, let’s go back to my question. How d..."
...,...,...,...
370,371,MCCAIN,"I’ve been involved, as I mentioned to you befo..."
371,372,OBAMA,"Well, let me just make a closing point. You kn..."
372,373,LEHRER,Few seconds. We’re almost finished.
373,374,MCCAIN,"Jim, when I came home from prison, I saw our v..."


In [ ]:
transcript_dataframe.tail(1)

In [ ]:
def add_counts(df):
    # Count the number of sentences by counting the number of periods, question marks, and exclamation points.
    df['Sentence_Count'] = df['Content'].apply(lambda x: x.count('.') + x.count('?') + x.count('!'))
    
    # Count the number of words by splitting the line into words and counting them.
    df['Word_Count'] = df['Content'].apply(lambda x: len(x.split()))

    return df

In [ ]:
transcript_dataframe = add_counts(transcript_dataframe)
transcript_dataframe

In [ ]:
transcript_dataframe.iloc[0:3]

In [12]:
def prompt_epl(row, prev_row=None):
    if not row['Content']: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Evaluate the following debate segment for Ethos (E), Pathos (P), and Logos (L) generation from 0.0 - 9.9 and a provide a brief explanation. It should be equally difficult to score 0.0 as it would be to score 9.9. Segment: '{row['Content']}'. Return a dictionary formatted like this: 'E: x.x, P: y.y, L: z.z, E_Expl: "", P_Expl: "", L_Expl: "". Explanations must be 1 sentence long."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt


In [13]:
def parse_content(content):
    # Use regular expressions to extract scores and explanations from content
    matches = re.findall(r'(\w+_?\w*): ([\d\.]+|".*?")', content)
    
    # Convert matches into a dictionary
    parsed_content = {key: value.strip('""') for key, value in matches}
    
    return parsed_content


In [14]:
def threadlock(idx, output_file_full, df, lock1, lock2, api_key):
    retries = 5
    for attempt in range(retries):
        try:
            # Get current row and previous row (if exists)
            row = df.loc[idx]
            prev_row = df.loc[idx - 1] if idx > 0 else None

            # Set the API key
            openai.api_key = api_key

            # Call OpenAI API and get response
            response = prompt_epl(row, prev_row)

            # Write the complete response to the JSON file
            with lock1:
                with open(output_file_full, 'a') as f:
                    json.dump(response, f)
                    f.write('\n')

            # Extract 'content' field and parse it to get scores and explanations
            content = response['choices'][0]['message']['content']
            parsed_content = parse_content(content)

            # Update DataFrame with new values (with thread lock to avoid race conditions)
            lock2.acquire()
            try:
                for key, value in parsed_content.items():
                    df.loc[idx, key] = value
            finally:
                lock2.release()

            # If we've reached this point, it means there were no exceptions, so we can break the loop
            break

        except Exception as e:
            # If it's the last attempt and it still fails, we print the error and move on to the next task
            if attempt == retries - 1:
                print(f'Error at index {idx} after {retries} attempts: {str(e)}')
            else:
                # If it's not the last attempt, we wait for a bit and try again
                time.sleep(2 ** attempt)

In [15]:
def get_epl(input_dataframe, output_file_full, startturn=None, endturn=None, runcount=None):
    # Error checking for parameters
    if startturn is not None and endturn is not None:
        if runcount is not None:
            raise ValueError("Parameter 'runcount' cannot be used together with 'startturn' and 'endturn'. Use 'runcount' alone or with either 'startturn' and 'endturn'.")

    # Determine start and end points for analysis
    start = startturn if startturn is not None else 0
    end = endturn if endturn is not None else len(input_dataframe)

    # Determine number of runs
    if runcount is not None:
        end = min(start + runcount, len(input_dataframe))

    # Check if 'end' exceeds the length of the speaker turns
    if end > len(input_dataframe):
        end = len(input_dataframe)
        print(f"Warning: The end point exceeded the length of the dataframe. It has been adjusted to the last turn: {end}")

    # Create locks for writing to file and dataframe
    lock1 = Lock()
    lock2 = Lock()

    # Use a ThreadPoolExecutor to process the lines concurrently
    with ThreadPoolExecutor(max_workers=25) as executor:
        # Assign API keys cyclically
        futures = [executor.submit(threadlock, i, output_file_full, input_dataframe, lock1, lock2, open_ai_keys[i % len(open_ai_keys)]) for i in range(start, end)]
        
        # Ensure all threads have finished
        for future in as_completed(futures):
            pass

    return input_dataframe

In [15]:
scored_df = get_epl(transcript_dataframe, "2008-all-responses-try-2.json")

In [ ]:
def get_total_cost(df):
    # Assuming df is the DataFrame obtained from the function process_jsonl_to_df

    # Calculate the total tokens
    total_prompt_tokens = df["usage.prompt_tokens"].sum()
    total_response_tokens = df["usage.completion_tokens"].sum()

    # Cost for 1k tokens
    prompt_token_price = 0.03  # This is the price per 1k prompt tokens
    response_token_price = 0.06  # This is the price per 1k response tokens

    # Calculate the total costs
    total_prompt_cost = (total_prompt_tokens / 1000) * prompt_token_price
    total_response_cost = (total_response_tokens / 1000) * response_token_price

    # Total cost
    total_cost = total_prompt_cost + total_response_cost

    # Create a summary dataframe
    summary = pd.DataFrame({
        "total_prompt_tokens": [total_prompt_tokens],
        "prompt_cost": [total_prompt_cost],
        "total_response_tokens": [total_response_tokens],
        "response_cost": [total_response_cost],
        "total_cost": [total_cost]
    })

    return summary